# CAD multi-asset VaR / ES case study
The reference portfolio holds XIU, two Canadian government bonds and a short USD/CAD straddle. We compare one-day loss models, backtest 750 forecasts and replay historical shocks on a common book.

The notebook opens from the repository root or the `notebooks/` directory. The committed report tables and figure are ready to display. Run the download and CLI commands in the README before enabling the full empirical rerun.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
reports = root / "reports"
display(Image(filename=str(root / "figures" / "risk-dashboard.png")))

## Compare risk in CAD
The Gaussian full-revaluation and delta-normal models use the same shock mean and covariance. Their difference isolates the effect of repricing under a common distribution, up to Monte Carlo quantile noise. Student-t changes the shock distribution as well.

In [ ]:
risk = pd.read_csv(reports / "snapshot.csv", index_col="method")
marks = pd.read_csv(reports / "marks.csv", index_col=0)
display(marks.round(2))
display(risk.round(0))
gap = 1 - risk.loc["delta_normal", "var_99"] / risk.loc["mc_normal", "var_99"]
print(f"Delta-normal VaR shortfall versus Gaussian full revaluation: {gap:.1%}")
paired = pd.read_csv(reports / "comparison.csv", index_col="portfolio")
display(paired.round(0))
display(pd.read_csv(reports / "sensitivities.csv").round({"value": 2}))

## Coverage and exception clustering
The first three p-values assess unconditional coverage, independence and both properties jointly. The traffic light uses the last 250 observations; the exception total uses all 750.

In [ ]:
diagnostics = pd.read_csv(reports / "diagnostics.csv", index_col="method")
display(diagnostics[["exceptions", "exception_rate", "kupiec_p", "independence_p",
                     "conditional_p", "exceptions_last_250", "traffic_last_250"]].round(4))

## Historical replays and hypothetical shocks
Every path revalues the reference holdings at constant maturity and strike. Worst close is selected at the portfolio level, so each component in that row uses the same date.

In [ ]:
stress = pd.read_csv(reports / "stress.csv")
legs = ["equity", "goc_5y", "goc_10y", "fx_straddle", "total"]
stress[legs] = (stress[legs] / 1000).round(1)
display(stress[["scenario", "measure", "scenario_date", *legs]].rename(
    columns={leg: f"{leg} (CAD thousands)" for leg in legs}))

## Optional full rerun
Set `rerun = True` after downloading the market data. This recalculates the model outputs with 10,000 Monte Carlo paths per distribution on each forecast date.

In [ ]:
rerun = False
if rerun:
    from var_es.analysis import study
    from var_es.market import load_market

    results = study(load_market(root / "data" / "market.csv"))
    display(results["daily_marks"].tail())
    display(results["backtest"].tail(10))